**Caroline Micheal Manjari**

**Registration Number : 24MDT1055**

**Problem Statement**

### **Optimizing Job Assignments: Finding the Best Match**  

Imagine you're a manager in charge of assigning four workers to four different tasks. Each worker has different skills, and each task has a different level of difficulty. Some workers are better suited for certain tasks, leading to higher productivity (or profit). Your goal is simple:  

- **If you want to maximize profit**, you need to assign workers to tasks in a way that gets the highest total efficiency.  
- **If you want to minimize cost**, you need to make assignments in a way that reduces expenses as much as possible.  

To make the best decisions, you have a **profit matrix**, which tells you how much value (profit) each worker generates when assigned to a specific task. The challenge? Each worker can only take on one task, and each task must be assigned to one worker.  

To solve this, we use **two smart techniques**:  

1. **Hungarian Method** – A fast algorithm that finds the best assignments automatically.  
2. **Integer Linear Programming (ILP)** – A mathematical approach that ensures the assignments follow all rules while maximizing profit or minimizing cost.  

At the end, you get the best possible assignment of workers to tasks, ensuring your team is working at peak efficiency while meeting your business goals. 🚀

In [1]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# Function for Hungarian Method (Maximization)
def hungarian_method_maximization(profit_matrix):
    # Cost matrix is the same as profit matrix for maximization
    cost_matrix = profit_matrix

    # Solve using Hungarian method (maximize profit)
    row_ind, col_ind = linear_sum_assignment(cost_matrix, maximize=True)

    total_profit = profit_matrix[row_ind, col_ind].sum()
    return total_profit, row_ind, col_ind

# Function for Hungarian Method (Minimization)
def hungarian_method_minimization(profit_matrix):
    # Convert profit maximization to cost minimization
    max_profit = profit_matrix.max()
    cost_matrix = max_profit - profit_matrix

    # Solve using Hungarian method (minimize cost)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    total_profit = profit_matrix[row_ind, col_ind].sum()
    return total_profit, row_ind, col_ind

In [2]:
!pip install pulp
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

# Function for ILP (Maximization)
def ilp_maximization(profit_matrix):
    num_workers, num_jobs = profit_matrix.shape
    prob = LpProblem("Job_Assignment_Maximization", LpMaximize)

    # Create binary decision variables (1 if worker i is assigned to job j, else 0)
    x = [[LpVariable(f"x_{i}_{j}", cat="Binary") for j in range(num_jobs)] for i in range(num_workers)]

    # Objective function (maximize total profit)
    prob += lpSum(profit_matrix[i][j] * x[i][j] for i in range(num_workers) for j in range(num_jobs))

    # Constraints: Each worker gets exactly one job
    for i in range(num_workers):
        prob += lpSum(x[i][j] for j in range(num_jobs)) == 1

    # Constraints: Each job is assigned to exactly one worker
    for j in range(num_jobs):
        prob += lpSum(x[i][j] for i in range(num_workers)) == 1

    # Solve the ILP problem
    prob.solve()
    total_profit = prob.objective.value()

    return total_profit, prob

# Function for ILP (Minimization)
def ilp_minimization(profit_matrix):
    num_workers, num_jobs = profit_matrix.shape
    prob = LpProblem("Job_Assignment_Minimization", LpMaximize)

    # Convert profit maximization to cost minimization
    max_profit = profit_matrix.max()
    cost_matrix = max_profit - profit_matrix

    # Create binary decision variables (1 if worker i is assigned to job j, else 0)
    x = [[LpVariable(f"x_{i}_{j}", cat="Binary") for j in range(num_jobs)] for i in range(num_workers)]

    # Objective function (minimize total cost)
    prob += lpSum(cost_matrix[i][j] * x[i][j] for i in range(num_workers) for j in range(num_jobs))

    # Constraints: Each worker gets exactly one job
    for i in range(num_workers):
        prob += lpSum(x[i][j] for j in range(num_jobs)) == 1

    # Constraints: Each job is assigned to exactly one worker
    for j in range(num_jobs):
        prob += lpSum(x[i][j] for i in range(num_workers)) == 1

    # Solve the ILP problem
    prob.solve()
    total_cost = prob.objective.value()

    return total_cost, prob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 48.0 MB/s eta 0:00:00


In [3]:
# Example Profit Matrix (for both maximization and minimization)
profit_matrix = np.array([
    [9, 2, 7, 8],
    [6, 4, 3, 7],
    [5, 8, 1, 8],
    [7, 6, 9, 4]
])

# **Maximization using Hungarian Method**
total_profit_max, row_ind_max, col_ind_max = hungarian_method_maximization(profit_matrix)
print(f"Maximization using Hungarian Method - Total Profit: {total_profit_max}")
print(f"Assignments: {list(zip(row_ind_max, col_ind_max))}")

# **Minimization using Hungarian Method**
total_profit_min, row_ind_min, col_ind_min = hungarian_method_minimization(profit_matrix)
print(f"Minimization using Hungarian Method - Total Profit: {total_profit_min}")
print(f"Assignments: {list(zip(row_ind_min, col_ind_min))}")

# **Maximization using ILP**
total_profit_ilp_max, prob_max = ilp_maximization(profit_matrix)
print(f"Maximization using ILP - Total Profit: {total_profit_ilp_max}")

# **Minimization using ILP**
total_cost_ilp_min, prob_min = ilp_minimization(profit_matrix)
print(f"Minimization using ILP - Total Cost: {total_cost_ilp_min}")

Maximization using Hungarian Method - Total Profit: 33
Assignments: [(np.int64(0), np.int64(0)), (np.int64(1), np.int64(3)), (np.int64(2), np.int64(1)), (np.int64(3), np.int64(2))]
Minimization using Hungarian Method - Total Profit: 33
Assignments: [(np.int64(0), np.int64(0)), (np.int64(1), np.int64(3)), (np.int64(2), np.int64(1)), (np.int64(3), np.int64(2))]
Maximization using ILP - Total Profit: 33.0
Minimization using ILP - Total Cost: 23.0
